## 本章演示大模型工具调用原理

In [1]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
# 将当前项目中env文件的环境变量加载进当前模块
load_dotenv()


# ==================== 1. 定义工具 ====================
def get_weather(location: str) -> str:
    return f"Current weather in {location} is sunny"

def square_root(x: float) -> float:
    return x ** 0.5

# 把函数放到dict中，方便根据函数名查找
tools_by_name = {"get_weather": get_weather, "square_root": square_root}

# ==================== 2. 定义工具的LLM描述 ====================
tools = [
    {
        # 工具类型是函数
        "type": "function",
        # 函数的描述
        "function": {
            # 函数名称
            "name": "get_weather",
            # 函数作用描述
            "description": "Get weather of a location, the user should supply a location first.",
            # 函数接收的参数
            "parameters": {
                "type": "object",
                "properties": {
                    # 第一个参数是location,类型是字符串
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "square_root",
            "description": "Calculate the square root of a given number.",
            "parameters": {
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The number to calculate square root for",
                    }
                },
                "required": ["x"]
            },
        }
    },
]

# ==================== 3. LLM客户端 ====================
client = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com"
)

# ==================== 4. 定义发送消息的方法 ====================
def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        tools=tools,
        extra_body={"thinking": {"type": "disabled"}}
    )
    return response.choices[0].message

# ==================== 5. 定义执行Tool调用的方法 ====================
def execute_tool(message, messages):
    """解析Tool信息，执行对应的函数，并将结果追加到消息列表"""
    tool_call = message.tool_calls[0]
    tool_id = tool_call.id
    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)

    print(f"调用工具: {tool_name}, 参数: {tool_args}")

    # 根据函数名找到对应函数并执行
    result = tools_by_name[tool_name](**tool_args)
    print(f"工具结果: {result}")

    # 将AI消息和Tool结果追加到消息列表
    messages.append(message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_id,
        "content": result
    })
    return messages

# ==================== 6. 主流程 ====================
# 准备用户消息列表
messages = [{"role": "user", "content": "杭州天气怎么样?"}]
print(f"User> {messages[0]['content']}")

# 第一次调用LLM，获取ToolCall
# 拿到大模型消息
model_message = send_messages(messages)
# 大模型想调用的工具是
print(f"Tool> {model_message.tool_calls[0]}")

# 执行Tool并将结果传回LLM
messages = execute_tool(model_message, messages)

# 再次调用LLM，生成最终答案
message = send_messages(messages)
print(f"AI> {message.content}")

User> 杭州天气怎么样?
Tool> ChatCompletionMessageFunctionToolCall(id='call_00_zmF53omYIe7Zgdan2V1o8804', function=Function(arguments='{"location": "杭州"}', name='get_weather'), type='function', index=0)
调用工具: get_weather, 参数: {'location': '杭州'}
工具结果: Current weather in 杭州 is sunny
AI> 杭州现在的天气是**晴**（sunny）☀️。

需要我帮你查询其他城市的天气，或者其他帮助吗？
